### Lesson 08 RAG Helper
Reusable RAG components for search, prompting, and generation

In [3]:
# Gemini version of the course RAG notebook: same pipeline, but OpenAI client points to Gemini's OpenAI-compatible endpoint.
import os # Needed to read the Gemini API key from environment variables
from dotenv import load_dotenv
load_dotenv()

from ingest import load_faq_data, build_index
from rag_helper import RAGBase
from openai import OpenAI

documents = load_faq_data()

index = build_index(documents)

openai_client = OpenAI(
    # Fetch the loaded Gemini key
    api_key=os.getenv("GEMINI_API_KEY"),
    
    # Reroute standard OpenAI traffic to Google's servers
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

answer = assistant.rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can still join, but if you want to receive a certificate, you need to submit your project while submissions are still being accepted.


In [2]:
custom_instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=custom_instructions,
)

answer = assistant.rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can still join. However, if you want to receive a certificate, you must submit your project while submissions are still being accepted. Note that you can only receive a certificate by finishing the course with a "live" cohort, as certificates are not awarded for self-paced mode.


In [6]:
assistant.rag("How do I get a certificate?")

'To get a certificate, you must follow these requirements:\n\n*   **Complete the course with a "live" cohort:** You cannot get a certificate in self-paced mode because peer-reviewing 3 capstone projects is required, and peer reviews can only happen while the course is actively running.\n*   **Pass the Capstone project:** While homework is recommended for reinforcing concepts and leaderboard points, it is not mandatory for the certificate; the Capstone project is.\n*   **Set your official name:** You should enter your official name (as it appears on your identification documents) in the "Edit Course Profile" section. If you do not change this, your certificate will be issued with your randomly assigned nickname (e.g., "Lucid Elbakyan").'

In [7]:
assistant.rag("Can I still join the course after it started?")

'Yes, you can still join the course after it has started. However, if you want to receive a certificate, you must submit your project while submissions are still being accepted.'

### Lesson 09 Data Ingestion
Reusable RAG components for search, prompting, and generation

In [8]:
# Connect to the existing SQLite search database (Run AFTER sqlite-ingest.ipnyb is run)
from sqlitesearch import TextSearchIndex

sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

In [9]:
# Check how many documents are in the index:
sqlite_index.count()

79

In [10]:
# Search the index for matching documents:
results = sqlite_index.search("Can I still join the course after it started?", num_results=5)
[doc["question"] for doc in results]

['I just discovered the course. Can I still join?',
 'I missed the first homework - can I still get a certificate?',
 'Do we submit 2 projects, what does attempt 1 and 2 mean?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'I am using Azure OpenAI and I am still getting an error of Error code: 400 - {\'error\': {\'message\': "Missing required parameter: \'tools[0].function\'.", \'type\': \'invalid_request_error\', \'param\': \'tools[0].function\', \'code\': \'missing_required_parameter\'}}?']

In [ ]:
# Use the existing SQLite index. Plug it into our modular code.

from rag_helper import RAGBase
from openai import OpenAI

openai_client = OpenAI(
    # Fetch the loaded Gemini key
    api_key=os.getenv("GEMINI_API_KEY"),
    
    # Reroute standard OpenAI traffic to Google's servers
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

assistant = RAGBase(
    index=sqlite_index, # replaced index with sqlite_index
    llm_client=openai_client,
)

In [11]:
# Test the persistent sqlite-backed index
answer = assistant.rag("Can I still join the course after it started?")
print(answer)

Yes, you can join the course after it has started. However, if you wish to receive a certificate, you must submit your project while submissions are still being accepted.


In [13]:
# Close the database connection to release resources and avoid keeping the file locked
sqlite_index.close()
print("Database connection closed.")

Database connection closed.
